# Milestone 3 — Machine Learning Model Development & Optimization

**Customer Churn Prediction — IBM Telco Dataset**


Sections:
1. Train / test split + SMOTE
2. Baseline training of 5 classifiers
3. 5-fold stratified cross-validation
4. Confusion matrices + classification reports
5. GridSearchCV tuning (Gradient Boosting & Random Forest)
6. Final comparison + ROC curves


## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.naive_bayes     import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_curve
)

from imblearn.pipeline       import Pipeline
from imblearn.over_sampling  import SMOTE

## 0. Load the cleaned dataset (output of Milestone 1)

In [ ]:
df = pd.read_csv('/content/telco_churn_cleaned.csv')
print('Loaded shape:', df.shape)
df.head()

## 1. Train / test split + SMOTE

Stratified 80/20 split preserves the 73/27 churn ratio in both subsets. SMOTE is applied **only to the training set** so the test set still reflects real-world distribution.

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Before SMOTE:')
print(y_train.value_counts())

In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('After SMOTE (training set):')
print(y_train_bal.value_counts())
print('\nTest set (unchanged):')
print(y_test.value_counts())
print('\nShapes -> X_train_bal:', X_train_bal.shape, '| X_test:', X_test.shape)

## 2. Baseline models — train and quick metrics

Five classifiers covering linear, ensemble, instance-based, and probabilistic approaches.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=42),
    'Random Forest':       RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Naive Bayes':         GaussianNB(),
}

In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train_bal, y_train_bal)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        'Model':    name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1':       round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':  round(roc_auc_score(y_test, y_proba), 4),
    })
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
print(results_df.to_string(index=False))

## 3. 5-fold Stratified Cross-Validation (SMOTE applied per fold)

Wraps SMOTE inside an `imblearn.Pipeline` so synthetic minority samples never leak into the validation fold.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []
for name, model in models.items():
    pipe = Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf',   model)
    ])

    scores_f1  = cross_val_score(pipe, X_train, y_train, cv=skf,
                                 scoring='f1', n_jobs=-1)
    scores_auc = cross_val_score(pipe, X_train, y_train, cv=skf,
                                 scoring='roc_auc', n_jobs=-1)

    cv_results.append({
        'Model':        name,
        'F1 mean':      round(scores_f1.mean(), 4),
        'F1 std':       round(scores_f1.std(), 4),
        'ROC-AUC mean': round(scores_auc.mean(), 4),
        'ROC-AUC std':  round(scores_auc.std(), 4),
    })

cv_df = pd.DataFrame(cv_results).sort_values('ROC-AUC mean', ascending=False)
print(cv_df.to_string(index=False))

## 4. Confusion matrices + classification reports

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['No Churn', 'Churn'])
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(name, fontsize=12, fontweight='bold')
    axes[i].grid(False)

for j in range(len(trained_models), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print('=' * 70)
print('CLASSIFICATION REPORTS')
print('=' * 70)
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred,
                                target_names=['No Churn', 'Churn'],
                                digits=3))

## 5. Hyperparameter tuning — GridSearchCV

Tune the two strongest baselines: Gradient Boosting and Random Forest.

### 5.1 Gradient Boosting

In [ ]:
gb_pipe = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf',   GradientBoostingClassifier(random_state=42))
])

gb_param_grid = {
    'clf__n_estimators':  [100, 200],
    'clf__learning_rate': [0.05, 0.1],
    'clf__max_depth':     [3, 5],
}

gb_grid = GridSearchCV(gb_pipe, gb_param_grid, cv=5,
                       scoring='roc_auc', n_jobs=-1, verbose=1)
gb_grid.fit(X_train, y_train)

print('--- Gradient Boosting tuned ---')
print('Best params:', gb_grid.best_params_)
print('Best CV ROC-AUC:', round(gb_grid.best_score_, 4))

### 5.2 Random Forest

In [ ]:
rf_pipe = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf',   RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    'clf__n_estimators':      [100, 200],
    'clf__max_depth':         [None, 20],
    'clf__min_samples_split': [2, 5],
}

rf_grid = GridSearchCV(rf_pipe, rf_param_grid, cv=5,
                       scoring='roc_auc', n_jobs=-1, verbose=1)
rf_grid.fit(X_train, y_train)

print('--- Random Forest tuned ---')
print('Best params:', rf_grid.best_params_)
print('Best CV ROC-AUC:', round(rf_grid.best_score_, 4))

## 6. Final comparison — baseline + tuned, on the held-out test set

In [ ]:
tuned_models = {
    'Gradient Boosting (tuned)': gb_grid.best_estimator_,
    'Random Forest (tuned)':     rf_grid.best_estimator_,
}
all_models = {**trained_models, **tuned_models}

final_results = []
for name, model in all_models.items():
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    final_results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1':        round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_proba), 4),
    })

final_df = (pd.DataFrame(final_results)
              .sort_values('ROC-AUC', ascending=False)
              .reset_index(drop=True))
print('=' * 80)
print('FINAL MODEL COMPARISON (sorted by ROC-AUC)')
print('=' * 80)
print(final_df.to_string(index=False))

### 6.1 ROC curves — all models on one chart

In [ ]:
plt.figure(figsize=(10, 8))
for name, model in all_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Pick the winner — by F1

ROC-AUC ranks the untuned GB at the top, but tuned GB has higher F1 and recall on the churn class — which matter more for a retention use case.

In [ ]:
final_by_f1 = final_df.sort_values('F1', ascending=False).reset_index(drop=True)
winner_name = final_by_f1.iloc[0]['Model']
winner_auc  = final_by_f1.iloc[0]['ROC-AUC']
winner_f1   = final_by_f1.iloc[0]['F1']
print(f'Best model (by F1): {winner_name}')
print(f'  Test ROC-AUC = {winner_auc} | F1 = {winner_f1}')

best_model = all_models[winner_name]